## ELEX 4653 Quiz 2 - Version 2

Instructions:

- Modify this notebook by adding the Python code described below.

- Test your code using the menu item `Cell  ► Run All` or `Run ► Run All Cells`

- Save the notebook (the .ipynb file) and upload it to the appropriate Assignment folder on the course web site.

_Version 2: corrected marking code for question 1._

### Question 1

Write a function `unitrand(n)` that returns a list of `n` psedo-random float numbers between 0 and 1 as generated by the `random.uniform()` function.  Your function may not call `random.seed()`. _Hint: the `random` module contains the `uniform()` function.

For example, `unitrand(3)` might return the three-element list `[0.38485411575081974, 0.23905300670762952, 0.38599879782365276]`.

In [2]:
def  unitrand(n):
    import random
    return [random.uniform(0,1) for i in range(n)]

unitrand(3)

[0.7464371777019373, 0.8832041482561763, 0.28255426185782384]

### Question 2

Write a function `excount(f,l)` that returns the number of times calling the function `f()` with an item from the list `l` raises an exception.

For example if the function `f()` calculates `1/x` then `excount(f,[1,0,2,0])` would return `2`.

In [3]:
def f(x):
    1/x

def excount(f,l):
    n  = 0
    for x in l:
        try:
            f(x)
        except:
            n += 1
    return n

excount(f,[1,0,2,0])        

2

### Question 3

Write a function `keyindex(l)` that returns a dictionary where the each item is generated from successive values taken from `l`.  The key of each item is the item's index in the list (starting with  1) and the value is the item from the list.   

For example `keyindex(["one", "two", "three"])` would return `{1:"one", 2:"two", 3:"three"}`.

In [4]:
def keyindex(l):
    return {k+1:v for k,v in enumerate(l)}

keyindex(["one", "two", "three"])        

{1: 'one', 2: 'two', 3: 'three'}

In [5]:
# lab validation code; do not modify
def labcheck(testing=0,ntest=10):
    '''
    Python exercise checking.
    Ed.Casas 2023-5-22
    Calls functions q<n>* and checks HMAC of return value[0].
    On mismatch prints return value[1] (function, arguments and return values).
    Setting testing=1 prints HMACs of correct results; paste into 'hashvalues'.
    Note:
    If q<n>* result not JSON-able, convert to string.
    Result order matters for comparison. Sort result if ordering not important.
    '''
    
    import base64, copy, hashlib, json, random, re, string, types 
    from random import randint, uniform
    import traceback
    
    # compare regex to strings that should/shouldn't match
    def checkre(pat,ok,nok):
        for s in ok:
            assert re.search(pat,s), \
                f"pattern '{pat}'\n did not match string '{s}'"
        for s in nok:
            assert not re.search(pat,s), \
                f"pattern '{pat}'\n matched string '{s}'"  
    
    # list of n words with nl letters from chars without repeats
    def randwords(n,chars=string.ascii_lowercase,nl=(2,5)):
        l = []
        while len(l)<n:
            w = ''.join([chars[randint(0,len(chars)-1)] for i in range(randint(*nl))])
            if w not in l:
                l.append(w)
        return l
    
    # convert sets to dicts and dict keys to strings so they can be sorted
    def orderkeys(o):
        if isinstance(o,set):
            return {str(k):None for k in o}
        if isinstance(o,dict):
            return {str(k):orderkeys(v) for k,v in o.items()}
        return o
    
    import math
    def roundn(num, n):
        return 0 if not num else round(num, -int(math.floor(math.log10(abs(num)))) + (n - 1))

    def ch(s,n=(1,1)):
        return randwords(1,chars=s,nl=n)[0]

    def Q1():
        f,a = unitrand,(randint(3,5),)
        oa = copy.deepcopy(a)
        r = repr(f(*a))
        return r,f"{f.__name__}({','.join([repr(s) for s in oa])}) returns {r}"  
        
    def Q2():
        def f(x):
            if randint(0,1):
                raise Exception
        f,a = excount,(f,[randint(0,5) for i in range(randint(3,5))])
        oa = copy.deepcopy(a)
        r = f(*a)
        return r,f"{f.__name__}({','.join([repr(s) for s in oa])}) returns {r}"
    
    def Q3():
        n=randint(5,7)
        s1=randwords(1,chars=string.ascii_lowercase,nl=(n,n))[0]
        f,a = keyindex,(s1,)
        oa = copy.deepcopy(a)
        r = f(*a)
        return r,f"{f.__name__}({','.join([repr(s) for s in oa])}) returns {r}"

    hashvalues = '''
VI+M4dCe+u8oFEX6dZ7lia1aoD/5+LDF27t2xzuM
qwW6ZEo+HrehHrehZEo+ZEo+6x5+Hreh6x5+ZEo+
U9ERrVrVLei/9bu71aRDi5L4LRE/fz/4kYdOeexZ
'''.split()

    newhash = ''
    dsize = 3 # HMAC base64 digest size (bytes, use 3 or 6 for 4 or 8 char digests)
    dlen = ((dsize*8+5)//6+3)//4*4
           
    for n,f in [(n,f) for n,f in locals().items() if callable(f) and re.search(r'^[Qq]\d+.*',n)]:
        random.seed(n)      
        hashes = '0'*dlen*ntest if testing else hashvalues.pop(0)
        err = ''
        while hashes and not err:
            h, hashes = hashes[:dlen], hashes[dlen:] 
            try:
                v,s = f()
                b = json.dumps(orderkeys(v),sort_keys=True).encode()
                c = base64.b64encode(hashlib.blake2b(b,digest_size=dsize).digest()).decode()
                if testing:
                    print(s)
                    newhash += c
                else:
                    if c != h:
                        err = f"Wrong result for test {n}: {s} (HMAC={c} instead of {h})"
            except Exception as e:
                traceback.print_exc()
                err = f"Error during test {n}: {e}"               
        if testing:
           newhash += '\n'
        else:
            print(err or f"Passed test {n}.")
            
    if testing:
        print(newhash)


labcheck(0)

Passed test Q1.
Passed test Q2.
Passed test Q3.
